# Sprint 4 Fine-Tuned Transformer Features

Thin Colab launcher for the `dl-final-project` Sprint 4 flow. Core logic lives in `src/` and `scripts/`.

This notebook is validation-only for Sprint 4. Do not compute test metrics here.

## Runtime

Use `Runtime > Change runtime type` and select a GPU before running full fine-tuning.

In [ ]:
!nvidia-smi

## Drive and Repository Setup

Expected Drive artifact root: `/content/drive/MyDrive/dl-final-artifact`.

The dataset split CSVs and HAM10000 images must resolve from the paths in the repo's dataset config or be copied/symlinked into the expected `data/` layout before running full jobs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = '/content/drive/MyDrive/dl-final-artifact'
REPO_URL = 'https://github.com/KasimDeliaci/dl-final-project.git'
REPO_DIR = '/content/dl-final-project'

!mkdir -p "$DRIVE_ROOT"
!if [ ! -d "$REPO_DIR/.git" ]; then git clone "$REPO_URL" "$REPO_DIR"; fi
%cd /content/dl-final-project
!git pull --ff-only
!git rev-parse --short HEAD

BUNDLE_CANDIDATES = [
    f'{DRIVE_ROOT}/ham10000_colab_bundle.tar',
    '/content/drive/MyDrive/ham10000_colab_bundle.tar',
    '/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar',
]
bundle = next((path for path in BUNDLE_CANDIDATES if __import__('pathlib').Path(path).exists()), None)
print('HAM10000 bundle:', bundle or 'not found')
if bundle:
    !tar -xf "$bundle" -C /content/dl-final-project
else:
    print('Upload/copy ham10000_colab_bundle.tar before full runs, or ensure data/ paths already exist.')

## Install Dependencies

In [ ]:
!pip -q install uv
!uv sync --dev

## Fast Verification

Run before the full GPU jobs.

In [ ]:
!PYTHONPATH=src uv run ruff check .
!PYTHONPATH=src uv run python -m pytest \
  tests/test_dataset_sprint1.py \
  tests/test_sprint2_features.py \
  tests/test_sprint3_fusion.py \
  tests/test_representation_complementarity.py \
  tests/test_sprint4_finetune.py

## Smoke Fine-Tuning Run

This checks the full code path without downloading pretrained weights or running the full dataset.

In [ ]:
!PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/finetune_backbones.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 \
  --epochs 1 \
  --batch-size 1 \
  --num-workers 0 \
  --limit-per-split 2 \
  --no-pretrained \
  --no-mixed-precision \
  --checkpoint-dir artifacts/checkpoints_smoke/ham10000/finetuned \
  --feature-root artifacts/features_smoke \
  --run-root artifacts/runs_smoke

## Full Sprint 4 Fine-Tuning

Run this only after dataset paths are valid in Colab. This produces train/validation fine-tuned feature caches and checkpoints. Test metrics are intentionally not computed.

In [ ]:
!PYTHONPATH=src uv run python scripts/finetune_backbone.py \
  --config configs/experiments/finetune_backbones.yaml \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --backbones vit_b16 swin_tiny beit_base \
  --batch-size 16 \
  --num-workers 2

## Fine-Tuned Single-Backbone MLP

In [ ]:
!PYTHONPATH=src uv run python scripts/train_mlp.py \
  --dataset-config configs/dataset/selected_dataset.yaml \
  --feature-source finetuned \
  --backbones vit_b16 swin_tiny beit_base \
  --batch-size 128

## Representative Fine-Tuned Fusion

In [ ]:
!PYTHONPATH=src uv run python scripts/run_fusion_matrix.py \
  --feature-source finetuned \
  --only-combination vit_b16 swin_tiny \
  --fusion-methods concat weighted_learned_512 weighted_pca_384 \
  --batch-size 128 \
  --run-tag s4_pair

!PYTHONPATH=src uv run python scripts/run_fusion_matrix.py \
  --feature-source finetuned \
  --only-combination vit_b16 swin_tiny beit_base \
  --fusion-methods concat weighted_learned_512 weighted_pca_384 \
  --batch-size 128 \
  --run-tag s4_triple

## Artifact Integrity Checks

These checks confirm that Sprint 4 artifacts are validation-only, aligned, finite, and have the expected row counts before interpreting plots.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch

FEATURE_ROOT = Path('artifacts/features/ham10000/finetuned')
RUN_ROOT = Path('artifacts/runs')
TABLE_ROOT = Path('artifacts/report_assets/tables')
FIGURE_ROOT = Path('artifacts/report_assets/figures')
EXPECTED_ROWS = {'train': 7008, 'val': 1504}

rows = []
for backbone in ['vit_b16', 'swin_tiny', 'beit_base']:
    for split, expected in EXPECTED_ROWS.items():
        path = FEATURE_ROOT / backbone / f'{split}.pt'
        if not path.exists():
            rows.append({'backbone': backbone, 'split': split, 'status': 'missing', 'path': str(path)})
            continue
        payload = torch.load(path, map_location='cpu', weights_only=False)
        features = payload['features']
        metadata = payload['metadata']
        rows.append({
            'backbone': backbone,
            'split': split,
            'rows': int(features.shape[0]),
            'expected_rows': expected,
            'feature_dim': int(features.shape[1]),
            'finite': bool(torch.isfinite(features).all()),
            'feature_source': metadata.get('feature_source'),
            'selection_metric': metadata.get('config', {}).get('selection_metric'),
            'test_policy': metadata.get('config', {}).get('test_policy'),
            'path': str(path),
        })
cache_checks = pd.DataFrame(rows)
display(cache_checks)

bad = cache_checks[(cache_checks.get('status') == 'missing') | (cache_checks.get('rows') != cache_checks.get('expected_rows')) | (cache_checks.get('finite') == False)]
if len(bad):
    print('Cache checks need attention before reporting results.')
else:
    print('All canonical fine-tuned cache checks passed.')

## Fine-Tuning Curves and Confusion Matrices

Each image-level fine-tuning run writes `training_curves.png`, `confusion_matrix.png`, `metrics_summary.csv`, `per_class_metrics.csv`, and `predictions.csv`.

In [ ]:
from IPython.display import Image, display, Markdown

finetune_runs = sorted(RUN_ROOT.glob('s4_finetune_*_seed42*'))
print('fine-tuning run folders:', len(finetune_runs))
for run_dir in finetune_runs:
    metrics_path = run_dir / 'metrics_summary.csv'
    if metrics_path.exists():
        display(Markdown(f'### {run_dir.name}'))
        display(pd.read_csv(metrics_path))
    for image_name in ['training_curves.png', 'confusion_matrix.png']:
        image_path = run_dir / image_name
        if image_path.exists():
            display(Markdown(f'**{image_name}**'))
            display(Image(filename=str(image_path)))

## Downstream Validation Tables

These tables are the main validation-only Sprint 4 interpretation surface. Compare against frozen baselines: ViT single `0.6924`, modest ViT+Swin+BEiT concat `0.6988`, and E2b ViT+Swin deep_reg `0.7262`.

In [ ]:
table_candidates = [
    TABLE_ROOT / 'single_backbone_finetuned_results.csv',
    TABLE_ROOT / 'single_backbone_finetuned_per_class_metrics.csv',
    TABLE_ROOT / 'finetuned_fusion_results.csv',
    TABLE_ROOT / 'finetuned_fusion_per_class_metrics.csv',
    TABLE_ROOT / 'finetuned_fusion_vs_single_validation.csv',
]
for path in table_candidates:
    display(Markdown(f'### {path}'))
    if path.exists():
        frame = pd.read_csv(path)
        if 'macro_f1' in frame.columns:
            frame = frame.sort_values('macro_f1', ascending=False)
        display(frame.head(20))
    else:
        print('missing')

## Report Figures

Show aggregate figures generated by the scripts. Missing figures usually mean the corresponding training/fusion command has not run yet.

In [ ]:
figure_candidates = [
    FIGURE_ROOT / 'finetuned_single_backbone_macro_f1.png',
    FIGURE_ROOT / 'finetuned_fusion_macro_f1.png',
]
for path in figure_candidates:
    display(Markdown(f'### {path}'))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('missing')

## Prediction Dump Checks

Downstream MLP/fusion prediction dumps should have exactly `1504` validation rows and one probability column per class.

In [ ]:
prediction_rows = []
for path in sorted(RUN_ROOT.glob('*/predictions.csv')):
    config_path = path.parent / 'run_config.json'
    if not config_path.exists():
        continue
    config = json.loads(config_path.read_text())
    if config.get('feature_source') != 'finetuned':
        continue
    frame = pd.read_csv(path)
    prediction_rows.append({
        'run_id': config.get('run_id'),
        'fusion_method': config.get('fusion_method'),
        'backbone': config.get('backbone'),
        'rows': len(frame),
        'prob_cols': len([column for column in frame.columns if column.startswith('prob_')]),
        'test_policy': config.get('test_policy'),
        'path': str(path),
    })
if prediction_rows:
    prediction_checks = pd.DataFrame(prediction_rows)
    sort_columns = [column for column in ['fusion_method', 'backbone', 'run_id'] if column in prediction_checks.columns]
    if sort_columns:
        prediction_checks = prediction_checks.sort_values(sort_columns)
    display(prediction_checks)
else:
    print('No finetuned prediction dumps found yet. Run the single-backbone MLP and/or fusion cells first.')

## Sync Generated Artifacts to Drive

Run after jobs complete. Generated checkpoints, features, and runs are intentionally not committed to Git.

In [ ]:
!mkdir -p "$DRIVE_ROOT/artifacts"
!rsync -av artifacts/ "$DRIVE_ROOT/artifacts/"